# PhishGuard: App Build & Validation

This notebook documents the implementation and testing of the PhishGuard Streamlit application. It covers URL feature extraction, validation against the cleaned dataset, and a smoke test of the trained machine learning model.

## 1. Setup

In [82]:
# !git clone https://github.com/umaimaa-shahid/PhishGuard.git
# %cd PhishGuard


In [83]:

# Set the project root as the working directory.
import os

os.chdir("/content/PhishGuard")

# Install the project dependencies.
!pip install -r requirements.txt
!pip install streamlit

# Confirm the working directory and active Git branch.
print("Working directory:", os.getcwd())
print("Dependencies installed.")

!git branch --show-current


Working directory: /content/PhishGuard
Dependencies installed.
main


## 2. Feature extractor (src/url_features.py)

In [ ]:
!cat src/url_features.py

import ipaddress
import re
from urllib.parse import urlparse
import pandas as pd


def extract_url_features(raw_url, dataset=None):
    if not isinstance(raw_url, str) or not raw_url.strip():
        raise ValueError("Please enter a valid URL.")

    url = raw_url.strip()
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url

    parsed = urlparse(url)
    host = parsed.hostname
    if not host:
        raise ValueError("Could not identify a hostname in this URL.")

    if dataset is not None and "URL" in dataset.columns:
        normalized = dataset["URL"].astype(str).str.strip()
        match = dataset.loc[normalized == raw_url.strip()]
        if not match.empty:
            return match.iloc[0:1].copy(), "Original dataset features"

    try:
        ipaddress.ip_address(host)
        is_ip = 1
    except ValueError:
        is_ip = 0

    host_parts = host.lower().split(".")
    if is_ip or len(host_parts) < 2:
        tld = ""
        subdomain_count = 0
        domain_body = host.lower()
    else:
        tld = host_parts[-1]
        subdomain_count = max(len(host_parts) - 2, 0)
        body_parts = host_parts[:-1]
        if body_parts and body_parts[0] == "www":
            body_parts = body_parts[1:]
        domain_body = ".".join(body_parts) if body_parts else host.lower()

    url_no_scheme = re.sub(r"^https?://", "", url, flags=re.IGNORECASE)
    url_len = max(len(url) - 1, 1)

    url_for_letters = re.sub(r"^www\.", "", url_no_scheme, flags=re.IGNORECASE)
    letters = max(sum(c.isalpha() for c in url_for_letters) - 1, 0)
    digits = sum(c.isdigit() for c in url_no_scheme)

    encoded = re.findall(r"%[0-9a-fA-F]{2}", url_no_scheme)
    encoded_count = len(encoded)

    raw_special = sum(
        not c.isalnum() and c not in {"=", "?", "&"}
        for c in url_no_scheme
    )
    other_special = max(raw_special - 1, 0)

    body_len = max(len(domain_body), 1)
    max_alpha = max((len(m) for m in re.findall(r"[A-Za-z]+", domain_body)), default=0)
    max_digit = max((len(m) for m in re.findall(r"[0-9]+", domain_body)), default=0)
    max_spec = max((len(m) for m in re.findall(r"[^A-Za-z0-9]+", domain_body)), default=0)
    continuation_rate = (max_alpha + max_digit + max_spec) / body_len

    features = {
        "URLLength": max(len(url) - 1, 0),
        "DomainLength": len(host),
        "IsDomainIP": is_ip,
        "TLD": tld,
        "CharContinuationRate": continuation_rate,
        "TLDLength": len(tld),
        "NoOfSubDomain": subdomain_count,
        "HasObfuscation": int(encoded_count > 0),
        "NoOfObfuscatedChar": encoded_count,
        "ObfuscationRatio": encoded_count / url_len,
        "NoOfLettersInURL": letters,
        "LetterRatioInURL": round(letters / url_len, 3),
        "NoOfDegitsInURL": digits,
        "DegitRatioInURL": round(digits / url_len, 3),
        "NoOfEqualsInURL": url_no_scheme.count("="),
        "NoOfQMarkInURL": url_no_scheme.count("?"),
        "NoOfAmpersandInURL": url_no_scheme.count("&"),
        "NoOfOtherSpecialCharsInURL": other_special,
        "SpacialCharRatioInURL": round(other_special / url_len, 3),
        "IsHTTPS": int(parsed.scheme.lower() == "https"),
    }

    return (
        pd.DataFrame([features]),
        "Calculated URL features"
    )


Overwriting src/url_features.py


In [85]:
import importlib
import src.url_features
importlib.reload(src.url_features)
from src.url_features import extract_url_features


## 3. Validate extractor against cleaned data

In [86]:

import pandas as pd
import numpy as np
import numbers
import joblib

from src.url_features import extract_url_features

# Load the cleaned dataset and trained model.
df = pd.read_csv("data/cleaned_with_domains.csv")
saved = joblib.load("models/final_model.joblib")

expected_features = saved["features"]

print("Dataset shape:", df.shape)
print("Model feature count:", len(expected_features))

# Select ten URLs using a fixed random seed for reproducibility.
sample = df.sample(n=min(10, len(df)), random_state=42)

comparison = []

# Calculate features without passing the dataset.
# This tests the extractor rather than reusing original dataset values.
for _, row in sample.iterrows():

    calculated, source = extract_url_features(row["URL"])

    for feature in expected_features:
        expected = row[feature]
        actual = calculated.iloc[0][feature]

        # Compare numeric and non-numeric values appropriately.
        if isinstance(expected, numbers.Number) and isinstance(actual, numbers.Number):
            matches = np.isclose(
                float(expected),
                float(actual),
                atol=1e-6
            )
        else:
            matches = str(expected).lower() == str(actual).lower()

        comparison.append({
            "URL": row["URL"],
            "Feature": feature,
            "Expected": expected,
            "Calculated": actual,
            "Match": bool(matches)
        })

# Convert the results into a DataFrame.
comparison_df = pd.DataFrame(comparison)

# Calculate agreement for each feature.
feature_accuracy = (
    comparison_df.groupby("Feature")["Match"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print("\nFEATURE VALIDATION RESULTS (%)")
display(feature_accuracy.to_frame("Match Percentage"))

# Calculate overall feature-level agreement.
match_percentage = round(
    comparison_df["Match"].mean() * 100, 2
)

print(f"\nOVERALL FEATURE MATCH: {match_percentage}%")

# Display individual comparisons for inspection.
print("\nDETAILED COMPARISON")
display(comparison_df)


Dataset shape: (235370, 27)
Model feature count: 20

FEATURE VALIDATION RESULTS (%)


,Match Percentage
Feature,
LetterRatioInURL,90.0
NoOfLettersInURL,90.0
URLLength,90.0
SpacialCharRatioInURL,90.0
HasObfuscation,100.0
IsDomainIP,100.0
DomainLength,100.0
CharContinuationRate,100.0
NoOfAmpersandInURL,100.0



OVERALL FEATURE MATCH: 98.0%

DETAILED COMPARISON


,URL,Feature,Expected,Calculated,Match
0,http://www.s107008.hostde20.fornex.host,URLLength,38,38,True
1,http://www.s107008.hostde20.fornex.host,DomainLength,32,32,True
2,http://www.s107008.hostde20.fornex.host,IsDomainIP,0,0,True
3,http://www.s107008.hostde20.fornex.host,TLD,host,host,True
4,http://www.s107008.hostde20.fornex.host,CharContinuationRate,0.565217,0.565217,True
...,...,...,...,...,...
195,https://www.omhof.org,NoOfQMarkInURL,0,0,True
196,https://www.omhof.org,NoOfAmpersandInURL,0,0,True
197,https://www.omhof.org,NoOfOtherSpecialCharsInURL,1,1,True
198,https://www.omhof.org,SpacialCharRatioInURL,0.05,0.05,True


## 4. Streamlit app (app.py)

In [ ]:

!cat app/app.py

# Import libraries for the Streamlit interface, model, and SHAP explanations.
import os
import joblib
import shap
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from src.url_features import extract_url_features


# Define paths to the saved model and cleaned dataset.
MODEL_PATH = "models/final_model.joblib"
DATA_PATH = "data/cleaned_with_domains.csv"


# Configure the Streamlit page.
st.set_page_config(
    page_title="PhishGuard",
    page_icon="🛡️",
    layout="centered"
)


# Load the trained model, dataset, and SHAP explainer once.
@st.cache_resource
def load_resources():

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError("Saved model was not found.")

    saved = joblib.load(MODEL_PATH)

    model = saved["model"]
    expected_features = saved["features"]

    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError("Cleaned dataset was not found.")

    dataset = pd.read_csv(DATA_PATH)

    background = dataset[expected_features].sample(
        n=min(50, len(dataset)),
        random_state=42
    )

    explainer = shap.Explainer(
        model.predict_proba,
        background
    )

    return model, expected_features, dataset, explainer


# Display the application introduction.
st.title("🛡️ PhishGuard")
st.subheader("Explainable Phishing URL Detection")

st.write(
    "Analyze a website URL using a trained machine learning model "
    "and understand which URL features influenced its prediction."
)

st.info(
    "PhishGuard analyzes URL text only. It does not visit websites "
    "or execute website content."
)

# Accept the URL entered by the user.
url = st.text_input(
    "Website URL",
    placeholder="https://example.com/login"
)


# Run URL analysis when the user clicks the button.
if st.button("Analyze URL", type="primary", use_container_width=True):

    if not url.strip():
        st.warning("Please enter a URL.")

    else:
        try:
            with st.spinner("Analyzing URL..."):

                # Load the model and extract the URL features.
                model, expected_features, dataset, explainer = load_resources()

                extracted, extraction_source = extract_url_features(
                    url,
                    dataset
                )

                # Arrange features in the exact order used during training.
                model_input = extracted.reindex(
                    columns=expected_features
                )

                if model_input.isnull().any().any():
                    missing = model_input.columns[
                        model_input.isnull().any()
                    ].tolist()

                    raise ValueError(
                        f"Missing model features: {missing}"
                    )

                # Predict the URL class and calculate class probabilities.
                prediction = int(model.predict(model_input)[0])

                probabilities = model.predict_proba(model_input)[0]

                classes = list(model.classes_)
                phishing_index = classes.index(1)
                legitimate_index = classes.index(0)

                phishing_prob = float(probabilities[phishing_index])
                legitimate_prob = float(probabilities[legitimate_index])

                # Generate a SHAP explanation for the phishing class.
                shap_result = explainer(model_input)

                phishing_shap = shap.Explanation(
                    values=shap_result.values[0, :, phishing_index],
                    base_values=shap_result.base_values[0, phishing_index],
                    data=model_input.iloc[0].values,
                    feature_names=expected_features
                )

            # Display the prediction and probability scores.
            st.divider()
            st.subheader("Result")
            if prediction == 1:
                st.error("⚠️ Prediction: PHISHING")
            else:
                st.success("✅ Prediction: LEGITIMATE")

            col1, col2 = st.columns(2)

            with col1:
                st.metric(
                    "Phishing probability",
                    f"{phishing_prob * 100:.2f}%"
                )

            with col2:
                st.metric(
                    "Legitimate probability",
                    f"{legitimate_prob * 100:.2f}%"
                )

            # Disclose whether the URL used original or calculated features.
            st.caption(f"Feature source: {extraction_source}")

            if "approximated" in extraction_source:
                st.warning(
                    "This URL was not found in the original dataset. "
                    "Some extracted features are approximations. "
                    "Treat this prediction as experimental."
                )

            # Display the SHAP feature-contribution explanation.
            st.divider()
            st.subheader("SHAP Explanation")

            st.write(
                "The waterfall plot shows how individual URL features "
                "contribute to the model's phishing-class output."
            )

            plt.figure(figsize=(10, 6))
            shap.plots.waterfall(phishing_shap, show=False)
            st.pyplot(plt.gcf())
            plt.close()

            # Display the values supplied to the trained model.
            st.divider()
            st.subheader("Extracted URL Features")

            st.dataframe(
                model_input.T.rename(columns={0: "Value"}),
                use_container_width=True
            )

        except Exception as error:
            st.error(f"Could not analyze this URL: {error}")


# Display the application footer.
st.divider()

st.caption(
    "PhishGuard | Explainable Phishing-URL Detection | "
    "Academic project prototype"
)


Overwriting app.py


## 5. Smoke test

In [88]:

# Import libraries and the feature extraction function.
import joblib
import pandas as pd

from src.url_features import extract_url_features

# Load the saved model and its expected feature columns.
saved = joblib.load("models/final_model.joblib")

model = saved["model"]
expected_features = saved["features"]

# Load the cleaned dataset.
df = pd.read_csv("data/cleaned_with_domains.csv")

# Select one URL from the dataset for the smoke test.
test_url = str(df.iloc[0]["URL"])

# Extract the URL features using the original dataset values.
features, source = extract_url_features(test_url, df)

# Arrange the extracted features in the model's expected order.
model_input = features.reindex(columns=expected_features)

# Run the model prediction and probability calculation.
prediction = model.predict(model_input)[0]
probabilities = model.predict_proba(model_input)[0]

# Display the test results.
print("Test URL:", test_url)
print("Feature source:", source)
print("Feature shape:", model_input.shape)
print("Missing values:", model_input.isnull().sum().sum())

print("Prediction:", "PHISHING" if prediction == 1 else "LEGITIMATE")

print("Probabilities:", dict(zip(model.classes_, probabilities)))

# Verify that the model received a complete feature row.
assert model_input.shape == (1, len(expected_features))
assert not model_input.isnull().any().any()

print("\nSUCCESS: Model prediction executed.")


Test URL: https://www.southbankmosaics.com
Feature source: Original dataset features
Feature shape: (1, 20)
Missing values: 0
Prediction: LEGITIMATE
Probabilities: {np.int64(0): np.float32(0.99658805), np.int64(1): np.float32(0.0034119226)}

SUCCESS: Model prediction executed.


## 6. Notes


- **Live application:** Deployment pending. The Streamlit Community Cloud URL will be added after deployment.

- **Feature validation:** The extractor achieved [ACTUAL RESULT]% overall agreement against the sampled cleaned dataset URLs (see Section 3).

- **Limitation:** CharContinuationRate is approximated for URLs not found in the original dataset. Predictions for these unseen URLs should therefore be treated as experimental.

- **Scope:** PhishGuard analyzes URL text only. It does not visit websites or inspect webpage content.

In [89]:
print("OVERALL MATCH:", match_percentage, "%")
display(feature_accuracy.to_frame("Match Percentage"))

OVERALL MATCH: 98.0 %


,Match Percentage
Feature,
LetterRatioInURL,90.0
NoOfLettersInURL,90.0
URLLength,90.0
SpacialCharRatioInURL,90.0
HasObfuscation,100.0
IsDomainIP,100.0
DomainLength,100.0
CharContinuationRate,100.0
NoOfAmpersandInURL,100.0


In [90]:
# Filter and display rows where calculations did not match expected dataset values
mismatches_df = comparison_df[~comparison_df['Match']]
display(mismatches_df)

,URL,Feature,Expected,Calculated,Match
160,http://www.apat.org,URLLength,19,18,False
170,http://www.apat.org,NoOfLettersInURL,7,6,False
171,http://www.apat.org,LetterRatioInURL,0.368,0.333,False
178,http://www.apat.org,SpacialCharRatioInURL,0.053,0.056,False


In [ ]:
import subprocess
import time
import requests

process = subprocess.Popen(
    [
        "streamlit", "run", "app/app.py",
        "--server.headless", "true",
        "--server.port", "8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for attempt in range(20):
    time.sleep(2)

    try:
        response = requests.get(
            "http://localhost:8501/_stcore/health",
            timeout=3
        )

        if response.status_code == 200:
            print("SUCCESS: Streamlit application started.")
            print("Health check:", response.text)
            break

    except requests.exceptions.RequestException:
        pass

else:
    print("Streamlit did not become ready.")
    process.terminate()

SUCCESS: Streamlit application started.
Health check: ok
